In [1]:
import pandas as pd

# ===== 1) Load files =====
base = "/Users/yichuan/Desktop/anime/AniVerse-Anime-Hybrid-Recommender/data"

anime = pd.read_csv(f"{base}/anime.csv", low_memory=False)
anime_syn = pd.read_csv(f"{base}/anime_with_synopsis.csv", low_memory=False)
anime_meta = pd.read_csv(f"{base}/anime_metadata_ready.csv", low_memory=False)

# ===== 2) Helper: find join key (MAL_ID / anime_id / etc.) =====
def find_key(df, preferred=("MAL_ID", "mal_id", "anime_id", "id")):
    for k in preferred:
        if k in df.columns:
            return k
    raise ValueError(f"No expected key found in columns: {df.columns.tolist()}")

k_anime = find_key(anime)
k_syn = find_key(anime_syn)
k_meta = find_key(anime_meta)

# Rename keys to a common name for safe merge
anime = anime.rename(columns={k_anime: "MAL_ID"})
anime_syn = anime_syn.rename(columns={k_syn: "MAL_ID"})
anime_meta = anime_meta.rename(columns={k_meta: "MAL_ID"})

# Optional: ensure same dtype on key
for d in (anime, anime_syn, anime_meta):
    d["MAL_ID"] = pd.to_numeric(d["MAL_ID"], errors="coerce")

# ===== 3) Drop duplicate MAL_ID rows before merge =====
anime = anime.drop_duplicates(subset=["MAL_ID"])
anime_syn = anime_syn.drop_duplicates(subset=["MAL_ID"])
anime_meta = anime_meta.drop_duplicates(subset=["MAL_ID"])

# ===== 4) Merge =====
# left join keeps all anime.csv rows
tmp = anime.merge(anime_syn, on="MAL_ID", how="left", suffixes=("", "_syn"))
anime_complete = tmp.merge(anime_meta, on="MAL_ID", how="left", suffixes=("", "_meta"))

# ===== 5) (Optional) Remove duplicate columns by exact same values =====
# If two columns have same info but different names, keep first one.
to_drop = []
cols = anime_complete.columns.tolist()
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        if c2 not in to_drop and anime_complete[c1].equals(anime_complete[c2]):
            to_drop.append(c2)
anime_complete = anime_complete.drop(columns=to_drop)

# ===== 6) Save =====
out_path = f"{base}/anime_complete.csv"
anime_complete.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)
print("Shape:", anime_complete.shape)
print("Columns:", len(anime_complete.columns))


Saved: /Users/yichuan/Desktop/anime/AniVerse-Anime-Hybrid-Recommender/data/anime_complete.csv
Shape: (17562, 43)
Columns: 43


In [3]:
import pandas as pd
import numpy as np

base = "/Users/yichuan/Desktop/anime/AniVerse-Anime-Hybrid-Recommender/data"

anime = pd.read_csv(f"{base}/anime.csv", low_memory=False)
meta = pd.read_csv(f"{base}/anime_metadata_ready.csv", low_memory=False)
syn = pd.read_csv(f"{base}/anime_with_synopsis.csv", low_memory=False)

def find_key(df, candidates=("MAL_ID", "mal_id", "anime_id", "id")):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No key column found in: {df.columns.tolist()}")

def find_synopsis_col(df):
    # 兼容 synopsis / sypnopsis
    for c in df.columns:
        if c.strip().lower() in ("synopsis", "sypnopsis"):
            return c
    return None

# 1) 统一主键
anime = anime.rename(columns={find_key(anime): "MAL_ID"})
meta = meta.rename(columns={find_key(meta): "MAL_ID"})
syn = syn.rename(columns={find_key(syn): "MAL_ID"})

# 2) 找 synopsis 列
meta_syn_col = find_synopsis_col(meta)
syn_syn_col = find_synopsis_col(syn)
if meta_syn_col is None or syn_syn_col is None:
    raise ValueError("Cannot find synopsis column in metadata or anime_with_synopsis")

meta = meta.rename(columns={meta_syn_col: "synopsis_meta"})
syn = syn.rename(columns={syn_syn_col: "synopsis_fallback"})

# 3) 清洗 key 和缺失值标记
for d in (anime, meta, syn):
    d["MAL_ID"] = pd.to_numeric(d["MAL_ID"], errors="coerce")

missing_tokens = ["", " ", "Unknown", "N/A", "NA", "None", "null", "NULL"]
meta["synopsis_meta"] = meta["synopsis_meta"].replace(missing_tokens, np.nan)
syn["synopsis_fallback"] = syn["synopsis_fallback"].replace(missing_tokens, np.nan)

# 4) 去重（同一个 MAL_ID 只留一行）
anime = anime.drop_duplicates(subset=["MAL_ID"])
meta = meta[["MAL_ID", "synopsis_meta"]].drop_duplicates(subset=["MAL_ID"])
syn = syn[["MAL_ID", "synopsis_fallback"]].drop_duplicates(subset=["MAL_ID"])

# 5) 合并：anime 主表 + metadata synopsis + fallback synopsis
anime_complete = (
    anime
    .merge(meta, on="MAL_ID", how="left")
    .merge(syn, on="MAL_ID", how="left")
)

# 6) 按优先级填充 synopsis
anime_complete["synopsis"] = anime_complete["synopsis_meta"].combine_first(anime_complete["synopsis_fallback"])

# 7) 清理中间列（可选）
anime_complete = anime_complete.drop(columns=["synopsis_meta", "synopsis_fallback"])

# 8) 保存
out_path = f"{base}/anime_complete.csv"
anime_complete.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)
print("Rows:", len(anime_complete))
print("Missing synopsis:", anime_complete["synopsis"].isna().sum())


Saved: /Users/yichuan/Desktop/anime/AniVerse-Anime-Hybrid-Recommender/data/anime_complete.csv
Rows: 17562
Missing synopsis: 8


In [2]:
import pandas as pd
import numpy as np

base = "/Users/yichuan/Desktop/anime/AniVerse-Anime-Hybrid-Recommender/data"

anime = pd.read_csv(f"{base}/anime.csv", low_memory=False)
meta = pd.read_csv(f"{base}/anime_metadata_ready.csv", low_memory=False)
syn = pd.read_csv(f"{base}/anime_with_synopsis.csv", low_memory=False)

def find_key(df, candidates=("MAL_ID", "mal_id", "anime_id", "id")):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No key column found in: {df.columns.tolist()}")

def find_synopsis_col(df):
    for c in df.columns:
        if c.strip().lower() in ("synopsis", "sypnopsis"):
            return c
    return None

# 1) 统一主键
anime = anime.rename(columns={find_key(anime): "MAL_ID"})
meta = meta.rename(columns={find_key(meta): "MAL_ID"})
syn = syn.rename(columns={find_key(syn): "MAL_ID"})

# 2) 找 synopsis 列
meta_syn_col = find_synopsis_col(meta)
syn_syn_col = find_synopsis_col(syn)
if meta_syn_col is None or syn_syn_col is None:
    raise ValueError("Cannot find synopsis column in metadata or anime_with_synopsis")

meta = meta.rename(columns={meta_syn_col: "synopsis_meta"})
syn = syn.rename(columns={syn_syn_col: "synopsis_fallback"})

# 3) 清洗 key 和缺失值标记
for d in (anime, meta, syn):
    d["MAL_ID"] = pd.to_numeric(d["MAL_ID"], errors="coerce")

missing_tokens = ["", " ", "Unknown", "N/A", "NA", "None", "null", "NULL"]
meta["synopsis_meta"] = meta["synopsis_meta"].replace(missing_tokens, np.nan)
syn["synopsis_fallback"] = syn["synopsis_fallback"].replace(missing_tokens, np.nan)

# 4) 要从 metadata 额外带过来的列（自动兼容大小写）
need_cols = ["Characters", "Staff"]
meta_col_map = {c.lower(): c for c in meta.columns}
extra_cols = [meta_col_map[c.lower()] for c in need_cols if c.lower() in meta_col_map]

if len(extra_cols) < len(need_cols):
    missing = [c for c in need_cols if c.lower() not in meta_col_map]
    print("Warning: these columns not found in metadata:", missing)

# 5) 去重（同一个 MAL_ID 只留一行）
anime = anime.drop_duplicates(subset=["MAL_ID"])
meta = meta[["MAL_ID", "synopsis_meta"] + extra_cols].drop_duplicates(subset=["MAL_ID"])
syn = syn[["MAL_ID", "synopsis_fallback"]].drop_duplicates(subset=["MAL_ID"])

# 6) 合并：anime 主表 + metadata(synopsis/characters/staff) + fallback synopsis
anime_complete = (
    anime
    .merge(meta, on="MAL_ID", how="left")
    .merge(syn, on="MAL_ID", how="left")
)

# 7) synopsis 按优先级填充
# 7) synopsis 按优先级填充：anime_with_synopsis 优先，缺失再用 metadata
anime_complete["synopsis"] = anime_complete["synopsis_fallback"].combine_first(anime_complete["synopsis_meta"])

# 8) 清理中间列
anime_complete = anime_complete.drop(columns=["synopsis_meta", "synopsis_fallback"])

# 9) 保存
out_path = f"{base}/anime_complete.csv"
anime_complete.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)
print("Rows:", len(anime_complete))
print("Missing synopsis:", anime_complete["synopsis"].isna().sum())
print("Added metadata columns:", extra_cols)


Saved: /Users/yichuan/Desktop/anime/AniVerse-Anime-Hybrid-Recommender/data/anime_complete.csv
Rows: 17562
Missing synopsis: 8
Added metadata columns: ['Characters', 'Staff']
